In [ ]:
# Repository setup for portable, repo-relative paths
from pathlib import Path
import sys

def _find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start

REPO_ROOT = _find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from dx_chat_entropy.paths import get_paths
PATHS = get_paths(REPO_ROOT)


# Differential LR Estimation

Estimate pairwise **differential likelihood ratios** for clinical findings
that discriminate between two diagnosis categories (`dx_cat_1` vs `dx_cat_2`).

Expected input workbook layout (per pair workbook):
- Row 0: sparse category labels (exactly 2 categories, forward-filled)
- Row 1: exemplar diseases for each category
- Rows >= 2: clinical findings in column 0

This notebook is designed to run after the pair-input build step
(`20_differential_build_inputs.ipynb`).


In [ ]:
from __future__ import annotations

import os

from dotenv import load_dotenv

from dx_chat_entropy.lr_differential_runner import build_runtime_config, run_differential

load_dotenv()


In [ ]:
# Runtime configuration (env-driven)
#
# Key environment variables:
# - DX_MODEL_ID
# - DX_API_BACKEND = responses|chat|auto
# - DX_REASONING_EFFORT = minimal|low|medium|high
# - DX_RESUME_MODE = recompute|skip_passing|repair_invalid
# - DX_MANIFEST_PATH (optional override)
# - DX_OUTPUTS_BY_MODEL_ROOT (optional override)
# - DX_INVALID_ROWS_PATH (optional override for repair mode)
#
# Defaults are repo-relative under data/processed/lr_differential.

RUNTIME_CONFIG = build_runtime_config(
    repo_root=REPO_ROOT,
    paths=PATHS,
    environ=os.environ,
)

print(
    {
        "model_id": RUNTIME_CONFIG.model_id,
        "api_backend": RUNTIME_CONFIG.api_backend,
        "reasoning_effort": RUNTIME_CONFIG.reasoning_effort,
        "resume_mode": RUNTIME_CONFIG.resume_mode,
        "manifest_path": str(RUNTIME_CONFIG.manifest_path),
        "outputs_by_model_root": str(RUNTIME_CONFIG.outputs_by_model_root),
        "invalid_rows_path": str(RUNTIME_CONFIG.invalid_rows_path),
        "max_pairs": RUNTIME_CONFIG.max_pairs,
        "max_findings": RUNTIME_CONFIG.max_findings,
    }
)


## Model and runtime controls

This notebook now delegates execution to a script-first runtime module (`dx_chat_entropy.lr_differential_runner`) so notebook and CLI runs share identical logic.

### What to change when switching models
- Usually only `DX_MODEL_ID` is required.
- Optionally adjust:
  - `DX_API_BACKEND` (`responses|chat|auto`)
  - `DX_REASONING_EFFORT` (`minimal|low|medium|high`)
  - `DX_VERBOSITY` (for models that support it)

### Resume behavior
- `DX_RESUME_MODE=recompute`: always recompute selected workbooks.
- `DX_RESUME_MODE=skip_passing`: skip workbook if existing output already passes structural LR audit checks.
- `DX_RESUME_MODE=repair_invalid`: patch only rows listed in invalid-rows manifest.

### Repair manifest path behavior
- If `DX_INVALID_ROWS_PATH` is unset, runtime first looks for:
  - `data/processed/lr_differential/manifests/invalid_rows_<MODEL_ID>.csv`
- Then falls back to:
  - `data/processed/lr_differential/manifests/invalid_rows.csv`

### Logging
Per-finding logs include scenario and comparator:
`scenario=<scenario_id> | comparator=<A> vs <B> | finding=<text>`


In [ ]:
# Differential-LR execution
#
# Core estimation logic is implemented in src/dx_chat_entropy/lr_differential_runner.py
# so notebook and script workflows stay behaviorally identical.

run_differential(
    config=RUNTIME_CONFIG,
    repo_root=REPO_ROOT,
)
